In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

results_dir = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf1_full_fiben_format_clean"
)

agg = pd.read_csv(results_dir / "benchmark_aggregate_results.csv")

if "official_id" not in agg.columns:
    agg["official_id"] = agg["query_name"].str.extract(r"^(IC\d+|IS\d+|INS\d+)")

def query_group_from_official_id(oid):
    if str(oid).startswith("IC"):
        return "complex_read"
    if str(oid).startswith("IS"):
        return "short_read"
    if str(oid).startswith("INS"):
        return "insert"
    return "other"

if "query_group" not in agg.columns:
    agg["query_group"] = agg["official_id"].apply(query_group_from_official_id)

hot = agg[agg["run_phase"] == "hot"].copy()

rows = []

for query_name, grp in hot.groupby("query_name"):
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]

    activated = grp[grp["final_benchmark_group"] != "control"].copy()
    primary = grp[grp["final_benchmark_group"] == "primary"].copy()

    best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]

    if len(primary) > 0:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        primary_regret = (
            best_primary["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"]
    else:
        best_primary = None
        primary_regret = np.nan

    # Full conceptual template space: G0-G9
    n_C = 10
    n_A = activated["candidate_id"].nunique()
    dsr = 1 - (n_A / n_C)

    rows.append({
        "scale_label": "sf1",
        "official_id": best_all["official_id"],
        "query_name": query_name,
        "query_group": best_all["query_group"],

        "n_tested_configs": grp["candidate_id"].nunique(),
        "n_activated_configs": n_A,
        "DSR": dsr,

        "best_config": best_all["g_class"],
        "best_group": best_all["final_benchmark_group"],
        "best_design_pattern": best_all["design_pattern"],
        "best_p95_ms": best_all["p95_latency_ms"],

        "top1_preserved_by_activated": (
            best_activated["candidate_id"] == best_all["candidate_id"]
        ),
        "activated_regret": (
            best_activated["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"],

        "best_primary_config": None if best_primary is None else best_primary["g_class"],
        "best_primary_p95_ms": None if best_primary is None else best_primary["p95_latency_ms"],
        "primary_regret": primary_regret,
    })

sf1_analysis_df = (
    pd.DataFrame(rows)
    .sort_values(["query_group", "official_id"])
    .reset_index(drop=True)
)

display(sf1_analysis_df)

print("Average DSR:", sf1_analysis_df["DSR"].mean())
print("Top-1 preservation activated:", sf1_analysis_df["top1_preserved_by_activated"].mean())
print("Mean activated regret:", sf1_analysis_df["activated_regret"].mean())
print("Mean primary regret:", sf1_analysis_df["primary_regret"].dropna().mean())

print("\nBest group counts:")
display(sf1_analysis_df["best_group"].value_counts())

print("\nSecondary affected winners:")
display(
    sf1_analysis_df[
        sf1_analysis_df["best_group"] == "secondary_affected"
    ][
        [
            "official_id",
            "query_name",
            "best_config",
            "best_design_pattern",
            "best_p95_ms",
            "best_primary_config",
            "best_primary_p95_ms",
            "primary_regret",
        ]
    ]
)

out_path = results_dir / "schemalens_reduction_analysis_hot.csv"
sf1_analysis_df.to_csv(out_path, index=False)

print("Saved:", out_path)

,scale_label,official_id,query_name,query_group,n_tested_configs,n_activated_configs,DSR,best_config,best_group,best_design_pattern,best_p95_ms,top1_preserved_by_activated,activated_regret,best_primary_config,best_primary_p95_ms,primary_regret
0,sf1,IC1,IC1_TransitiveFriendsWithName,complex_read,2,2,0.8,G3,primary,root_with_references_or_summaries,227.727482,True,0.00000,G3,227.727482,0.000000
1,sf1,IC2,IC2_RecentMessagesByFriends,complex_read,2,2,0.8,G3,primary,root_with_references_or_summaries,37.090559,True,0.00000,G3,37.090559,0.000000
2,sf1,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,complex_read,4,4,0.6,G7,secondary_affected,containment_baseline,191.817495,True,0.00000,G3,196.627160,0.025074
3,sf1,IC4,IC4_NewTopics,complex_read,2,2,0.8,G0,primary,root_with_references,66.387148,True,0.00000,G0,66.387148,0.000000
4,sf1,IC5,IC5_NewGroups,complex_read,6,6,0.4,G6,secondary_affected,referenced_or_reverse_indexed_edges,176.753622,True,0.00000,G0,185.673138,0.050463
5,sf1,IC6,IC6_TagCoOccurrence,complex_read,2,2,0.8,G3,primary,root_with_references_or_summaries,198.528747,True,0.00000,G3,198.528747,0.000000
6,sf1,IC7,IC7_RecentLikers,complex_read,4,4,0.6,G3,primary,root_with_references_or_summaries,7.009322,True,0.00000,G3,7.009322,0.000000
7,sf1,INS1,INS1_AddPerson,insert,2,2,0.8,G3,primary,root_with_references_or_summaries,1.624116,True,0.00000,G3,1.624116,0.000000
8,sf1,INS2,INS2_AddLikeToPost,insert,2,2,0.8,G6,primary,referenced_or_reverse_indexed_edges,1.159713,True,0.00000,G6,1.159713,0.000000
9,sf1,INS3,INS3_AddLikeToComment,insert,2,2,0.8,G6,primary,referenced_or_reverse_indexed_edges,1.868959,True,0.00000,G6,1.868959,0.000000


Average DSR: 0.7136363636363637
Top-1 preservation activated: 0.9545454545454546
Mean activated regret: 0.023536810663730146
Mean primary regret: 0.033818552363028075

Best group counts:


best_group
primary               15
secondary_affected     6
control                1
Name: count, dtype: int64


Secondary affected winners:


,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
2,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,G7,containment_baseline,191.817495,G3,196.627160,0.025074
4,IC5,IC5_NewGroups,G6,referenced_or_reverse_indexed_edges,176.753622,G0,185.673138,0.050463
12,INS6,INS6_AddPost,G9,hybrid_containment,2.188292,G3,2.343452,0.070905
13,INS7,INS7_AddComment,G9,hybrid_containment,2.159610,G3,2.172902,0.006154
16,IS2,IS2_RecentMessagesOfPerson,G9,hybrid_containment,3.230855,None,NaN,NaN
21,IS7,IS7_RepliesOfMessage,G9,hybrid_containment,11.009079,G0,11.447059,0.039784


Saved: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf1_full_fiben_format_clean/schemalens_reduction_analysis_hot.csv
